In [756]:
%load_ext autoreload
%autoreload 2

import os

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from files import data, features, models

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [757]:
# Make sure this matches project root every time

ROOT = '..'

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(ROOT)

os.environ["DATA_DIR"] = "datasets"

In [758]:
listings = data.load("listings")

list_aware = data.load("list_aware")
off_market = data.load("off_market")

In [972]:
homes = listings.sort_values(by = 'date_sold', ascending = True).reset_index(drop = True)

homes.head(3)

,ml_num,sold_price,list_price,date_listed,date_sold,days_on_market,house_type_name,house_style,bedrooms,washrooms,...,house_area,lot_front,lot_depth,municipality,community,latitude,longitude,external_avm_estimate,external_avm_estimate_date,remarks
0,SYN-0023247,364100.0,371200.0,2020-12-29,2021-01-01,3,Condo Apt,Apartment,1,1,...,622,NaN,NaN,Toronto,Sumacpark,43.602039,-79.261184,366200.0,2021-02-14,Looking For More Space In A Turnkey Condo? Thi...
1,SYN-0124536,529900.0,514800.0,2020-11-27,2021-01-01,35,Condo Apt,Apartment,2,1,...,600-799,NaN,NaN,Toronto,Hickoryhaven,43.656349,-79.536538,NaN,2020-12-31,Looking for a downtown commuter condo with pot...
2,SYN-0024710,1805700.0,1745700.0,2020-12-19,2021-01-01,13,Detached,Bungalow,4,2,...,2100-2299,25.14,137.68,Toronto,Hickorymills,43.668013,-79.493774,1790700.0,2021-11-28,A bright eat-in kitchen sets the stage for ent...


In [973]:
cat_cols = ['house_type_name', 'municipality', 'house_style', 'community']

for col in cat_cols:
    homes[col] = pd.Categorical(homes[col], categories = homes[col].unique())

In [974]:
homes['sold_to_list_price_ratio'] = homes['sold_price'] / homes['list_price']

homes['log_list_price'] = np.log(homes['list_price'])
homes['log_sold_price'] = np.log(homes['sold_price'])

In [975]:
amenity_cols = ['bedrooms', 'washrooms', 'parking_total', 'garage']

homes['num_amenities'] = homes[amenity_cols].sum(axis = 1)

In [976]:
homes['house_area_sqft'] = features.square_feet(homes['house_area'])

homes['list_price_per_sqft'] = homes['list_price'] / homes['house_area_sqft']
homes['sold_price_per_sqft'] = homes['sold_price'] / homes['house_area_sqft']

homes['lot_size'] = homes['lot_front'] * homes['lot_depth']

homes['lot_to_house_area'] = homes['lot_size'] / homes['house_area_sqft']
homes['bedrooms_per_sqft'] = homes['bedrooms'] / homes['house_area_sqft']
homes['washrooms_per_sqft'] = homes['washrooms'] / homes['house_area_sqft']
homes['washrooms_to_bedrooms'] = homes['washrooms'] / homes['bedrooms'].replace(0, np.nan)
homes['garage_to_parking'] = homes['garage'] / homes['parking_total'].replace(0, np.nan)

In [977]:
homes['avm_estimate'] = homes[homes['external_avm_estimate_date'] < homes['date_sold']]['external_avm_estimate']

In [978]:
homes['list_month_yr'] = homes['date_listed'].dt.to_period('M')
homes['sold_month_yr'] = homes['date_sold'].dt.to_period('M')

In [979]:
earliest_list_date = homes['date_listed'].min()
mean_days_per_month = ((365 * 4) + (366 * 2)) / (12 * 6)

homes['days_since_list_start'] = (homes['date_listed'] - earliest_list_date).dt.days
homes['months_since_list_start'] = homes['days_since_list_start'] / mean_days_per_month

In [980]:
homes['list_month'] = homes['date_listed'].dt.month
homes['list_quart'] = homes['date_listed'].dt.quarter

homes['list_month_sin'] = np.sin(2 * np.pi * homes['list_month'] / 12)
homes['list_month_cos'] = np.cos(2 * np.pi * homes['list_month'] / 12)

homes['list_month_2sin'] = np.sin(4 * np.pi * homes['list_month'] / 12)
homes['list_month_2cos'] = np.cos(4 * np.pi * homes['list_month'] / 12)

homes['list_quart_sin'] = np.sin(2 * np.pi * homes['list_quart'] / 4)
homes['list_quart_cos'] = np.cos(2 * np.pi * homes['list_quart'] / 4)

In [981]:
tor_lon_scale = 0.72

homes['longitude'] *= tor_lon_scale

for deg in [15, 30, 45, 60, 75, 105, 120, 135, 150, 165]:
    rad = np.deg2rad(deg)
    homes[f'lat_lon_{deg}'] = (np.cos(rad) * homes['latitude'] + np.sin(rad) * homes['longitude'])

In [982]:
homes['remarks_count'] = homes['remarks'].str.count(r"\b\w+\b")

remark_kwds = {"renovated": r"renovat|updated|modern|upgrad|remodel",
               "needs_work": r"needs work|fixer|as.is|tlc|original condition",
               "transit": r"subway|transit|t.tc|go station",
               "basement": r"basement|lower level",
               "ravine": r"ravine",
               "view": r"unobstructed|panoramic|city view",
               "balcony": r"balcony|terrace",
               "locker": r"locker|storage unit",
               "move_in": r"move.in ready|turnkey",
               }

for kwd, pattern in remark_kwds.items():
    homes[f'has_{kwd}'] = homes['remarks'].str.lower().str.contains(pattern, regex = True).astype(int)

In [983]:
homes['date_listed_before_1d'] = homes['date_listed'] - pd.DateOffset(days = 1)
homes['date_sold_before_1d'] = homes['date_sold'] - pd.DateOffset(days = 1)

In [984]:
homes.head(3)

,ml_num,sold_price,list_price,date_listed,date_sold,days_on_market,house_type_name,house_style,bedrooms,washrooms,...,has_needs_work,has_transit,has_basement,has_ravine,has_view,has_balcony,has_locker,has_move_in,date_listed_before_1d,date_sold_before_1d
0,SYN-0023247,364100.0,371200.0,2020-12-29,2021-01-01,3,Condo Apt,Apartment,1,1,...,0,0,0,1,1,0,0,1,2020-12-28,2020-12-31
1,SYN-0124536,529900.0,514800.0,2020-11-27,2021-01-01,35,Condo Apt,Apartment,2,1,...,1,0,0,0,0,1,0,0,2020-11-26,2020-12-31
2,SYN-0024710,1805700.0,1745700.0,2020-12-19,2021-01-01,13,Detached,Bungalow,4,2,...,0,0,0,0,0,0,0,1,2020-12-18,2020-12-31


In [987]:
groupings = ['house_type_name', 'municipality', 'house_style']

daily_sales = homes.groupby(['date_sold'] + groupings,
                            observed = False).agg({'ml_num': 'count',
                                                   'log_sold_price': 'sum',
                                                   'days_on_market': 'sum'}).reset_index().rename(columns = {'date_sold': 'date_sold_last'})

groupings_obj = daily_sales.groupby(groupings, observed = True)

In [988]:
daily_sales.head(3)

,date_sold_last,house_type_name,municipality,house_style,ml_num,log_sold_price,days_on_market
0,2021-01-01,Condo Apt,Toronto,Apartment,31,414.943441,642
1,2021-01-01,Condo Apt,Toronto,Bungalow,0,0.000000,0
2,2021-01-01,Condo Apt,Toronto,2-Storey,1,13.126555,88


In [989]:
cat_offset = len(pd.MultiIndex.from_product([daily_sales[col].cat.categories for col in groupings]))

daily_sales['num_sales'] = groupings_obj['ml_num'].cumsum()
daily_sales['cumsum_log_sold_price'] = groupings_obj['log_sold_price'].cumsum()
daily_sales['cumsum_days_on_market'] = groupings_obj['days_on_market'].cumsum()

for w in [1, 30, 90]:
    daily_sales[f'num_sales_lag_{w}d'] = daily_sales['num_sales'].shift(w * cat_offset)
    daily_sales[f'cumsum_log_sold_price_lag_{w}d'] = daily_sales['cumsum_log_sold_price'].shift(w * cat_offset)
    daily_sales[f'cumsum_days_on_market_lag_{w}d'] = daily_sales['cumsum_days_on_market'].shift(w * cat_offset)

    daily_sales[f'diff_num_sales_{w}d'] = (daily_sales['num_sales'] - daily_sales[f'num_sales_lag_{w}d']).replace(0, np.nan)

    daily_sales[f'diff_log_sold_price_{w}d'] = daily_sales['cumsum_log_sold_price'] - daily_sales[f'cumsum_log_sold_price_lag_{w}d']
    daily_sales[f'mean_log_sold_price_{w}d'] = daily_sales[f'diff_log_sold_price_{w}d'] / daily_sales[f'diff_num_sales_{w}d']

    daily_sales[f'diff_days_on_market_{w}d'] = daily_sales['cumsum_days_on_market'] - daily_sales[f'cumsum_days_on_market_lag_{w}d']
    daily_sales[f'mean_days_on_market_{w}d'] = daily_sales[f'diff_days_on_market_{w}d'] / daily_sales[f'diff_num_sales_{w}d']

In [992]:
daily_sales.head(3)

,date_sold_last,house_type_name,municipality,house_style,ml_num,log_sold_price,days_on_market,num_sales,cumsum_log_sold_price,cumsum_days_on_market,...,diff_days_on_market_30d,mean_days_on_market_30d,num_sales_lag_90d,cumsum_log_sold_price_lag_90d,cumsum_days_on_market_lag_90d,diff_num_sales_90d,diff_log_sold_price_90d,mean_log_sold_price_90d,diff_days_on_market_90d,mean_days_on_market_90d
0,2021-01-01,Condo Apt,Toronto,Apartment,31,414.943441,642,31,414.943441,642,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-01-01,Condo Apt,Toronto,Bungalow,0,0.000000,0,0,0.000000,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-01-01,Condo Apt,Toronto,2-Storey,1,13.126555,88,1,13.126555,88,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [993]:
daily_sales.tail(3)

,date_sold_last,house_type_name,municipality,house_style,ml_num,log_sold_price,days_on_market,num_sales,cumsum_log_sold_price,cumsum_days_on_market,...,diff_days_on_market_30d,mean_days_on_market_30d,num_sales_lag_90d,cumsum_log_sold_price_lag_90d,cumsum_days_on_market_lag_90d,diff_num_sales_90d,diff_log_sold_price_90d,mean_log_sold_price_90d,diff_days_on_market_90d,mean_days_on_market_90d
948957,2026-05-31,Co-Op Apt,York,Sidesplit 4,0,0.0,0,0,0.0,0,...,0.0,NaN,0.0,0.0,0.0,NaN,0.0,NaN,0.0,NaN
948958,2026-05-31,Co-Op Apt,York,1 Storey/Apt,0,0.0,0,0,0.0,0,...,0.0,NaN,0.0,0.0,0.0,NaN,0.0,NaN,0.0,NaN
948959,2026-05-31,Co-Op Apt,York,Backsplit 5,0,0.0,0,0,0.0,0,...,0.0,NaN,0.0,0.0,0.0,NaN,0.0,NaN,0.0,NaN


In [994]:
valid_cols = homes.columns  # Important, define validation-only columns before merging homes with time-based features

homes = homes.merge(daily_sales.drop(columns = ['ml_num', 'log_sold_price', 'days_on_market']),
                    how = 'left', left_on = ['date_listed_before_1d'] + groupings, right_on = ['date_sold_last'] + groupings)

In [995]:
homes.head(3)

,ml_num,sold_price,list_price,date_listed,date_sold,days_on_market,house_type_name,house_style,bedrooms,washrooms,...,diff_days_on_market_30d,mean_days_on_market_30d,num_sales_lag_90d,cumsum_log_sold_price_lag_90d,cumsum_days_on_market_lag_90d,diff_num_sales_90d,diff_log_sold_price_90d,mean_log_sold_price_90d,diff_days_on_market_90d,mean_days_on_market_90d
0,SYN-0023247,364100.0,371200.0,2020-12-29,2021-01-01,3,Condo Apt,Apartment,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SYN-0124536,529900.0,514800.0,2020-11-27,2021-01-01,35,Condo Apt,Apartment,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SYN-0024710,1805700.0,1745700.0,2020-12-19,2021-01-01,13,Detached,Bungalow,4,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [996]:
homes.tail(3)

,ml_num,sold_price,list_price,date_listed,date_sold,days_on_market,house_type_name,house_style,bedrooms,washrooms,...,diff_days_on_market_30d,mean_days_on_market_30d,num_sales_lag_90d,cumsum_log_sold_price_lag_90d,cumsum_days_on_market_lag_90d,diff_num_sales_90d,diff_log_sold_price_90d,mean_log_sold_price_90d,diff_days_on_market_90d,mean_days_on_market_90d
149997,SYN-0057620,607300.0,622700.0,2026-05-13,2026-05-31,18,Condo Apt,Apartment,2,2,...,7875.0,43.994413,12854.0,172112.351016,337091.0,475.0,6294.657947,13.251911,20124.0,42.366316
149998,SYN-0140519,985300.0,954800.0,2026-05-24,2026-05-31,7,Semi-Detached,Bungalow,3,2,...,75.0,10.714286,469.0,6507.097012,6419.0,24.0,333.233418,13.884726,252.0,10.500000
149999,SYN-0023188,704200.0,709800.0,2026-05-03,2026-05-31,28,Condo Apt,Apartment,2,2,...,6847.0,38.903409,12811.0,171540.677683,334969.0,453.0,6007.168824,13.260858,18847.0,41.604857


In [997]:
list_cols = ['list_price', 'list_price_per_sqft']

loc_cols = [col for col in homes if col.startswith('lat_lon_')]

remark_cols = ['remarks_count'] + [col for col in homes if col.startswith('has_')]

time_angle_cols = ['list_month_sin', 'list_month_cos']
# time_angle_cols = ['list_month_sin', 'list_month_cos', 'list_month_2sin', 'list_month_2cos', 'list_quart_sin', 'list_quart_cos']

time_cols = ['list_month', 'list_quart']

list_aware_cols = (list_cols + cat_cols + loc_cols + remark_cols + amenity_cols + time_angle_cols +
                   ['lot_front', 'lot_depth'] + ['house_area_sqft', 'lot_size', 'months_since_list_start'])

off_market_cols = (cat_cols + loc_cols + remark_cols + amenity_cols + time_cols +
                   ['lot_front', 'lot_depth'] + ['house_area_sqft', 'lot_size', 'months_since_list_start'] +
                   ['lot_to_house_area', 'bedrooms_per_sqft', 'washrooms_per_sqft', 'washrooms_to_bedrooms', 'garage_to_parking'])

# moving_cols = []
moving_cols = ['mean_log_sold_price_1d', 'mean_log_sold_price_30d', 'mean_log_sold_price_90d']
# moving_cols = ['mean_log_sold_price_30d']
# moving_cols = ['mean_log_sold_price_1d', 'mean_log_sold_price_30d', 'mean_days_on_market_1d', 'mean_days_on_market_30d']

In [946]:
results = []

month_periods = pd.period_range("2025-12", "2026-05", freq = "M")

for month in month_periods:
    train, valid = homes[homes['sold_month_yr'] < month], homes[homes['sold_month_yr'] == month][valid_cols]

    X_train, y_train = train[list_aware_cols], train['sold_to_list_price_ratio']
    X_valid, y_valid = valid[list_aware_cols], valid['sold_price']

    gb_model = models.HistGradientBoostingRegressor(random_state = 0,
                                                    loss = 'absolute_error', learning_rate = 0.06,
                                                    max_iter = 240, max_leaf_nodes = 30, min_samples_leaf = 40, max_depth = None,
                                                    l2_regularization = 0.1, max_bins = 255, validation_fraction = None,
                                                    max_features = 1., early_stopping = False, verbose = 1)

    gb_model.fit(X_train, np.log(y_train))

    preds = np.exp(gb_model.predict(X_valid)) * X_valid['list_price']
    results.append(pd.DataFrame({'predicted': preds, 'y': y_valid, 'house_type': X_valid['house_type_name']}))
    print('---')

print("List-aware training complete.")

results = pd.concat(results).reset_index(drop = True)

Binning 0.045 GB of training data: 0.306 s
Fitting gradient boosted rounds:
Fit 240 trees in 16.161 s, (7200 total leaves)
Time spent computing histograms: 6.307s
Time spent finding best splits:  1.766s
Time spent applying splits:      1.682s
Time spent predicting:           0.212s
---
Binning 0.045 GB of training data: 0.365 s
Fitting gradient boosted rounds:
Fit 240 trees in 23.159 s, (7200 total leaves)
Time spent computing histograms: 9.418s
Time spent finding best splits:  3.083s
Time spent applying splits:      3.118s
Time spent predicting:           0.241s
---
Binning 0.046 GB of training data: 0.330 s
Fitting gradient boosted rounds:
Fit 240 trees in 21.903 s, (7200 total leaves)
Time spent computing histograms: 8.865s
Time spent finding best splits:  2.831s
Time spent applying splits:      2.910s
Time spent predicting:           0.240s
---
Binning 0.046 GB of training data: 0.371 s
Fitting gradient boosted rounds:
Fit 240 trees in 19.904 s, (7200 total leaves)
Time spent compu

In [947]:
print("MdA%E: " + str(100 * ((results['predicted'] - results['y']).abs() / results['y']).median()))
print("MA%E: " + str(100 * ((results['predicted'] - results['y']).abs() / results['y']).mean()))
print("Signed MdA%E: " + str(100 * ((results['predicted'] - results['y']) / results['y']).median()))
print("Signed MA%E: " + str(100 * ((results['predicted'] - results['y']) / results['y']).mean()))

print('---')

for ht in results['house_type'].unique():
    results_ht = results[results['house_type'] == ht]

    print(f"MdA%E {ht}: " + str(100 * ((results_ht['predicted'] - results_ht['y']).abs() / results_ht['y']).median()))
    print(f"MA%E {ht}: " + str(100 * ((results_ht['predicted'] - results_ht['y']).abs() / results_ht['y']).mean()))

print('---')

print("Baseline MdA%E list-aware: 7.3")
print("Baseline MA%E list_aware: 8.1")

MdA%E: 4.124959146064726
MA%E: 5.238497042351922
Signed MdA%E: 0.2717601914774896
Signed MA%E: -0.14558143236773327
---
MdA%E Detached: 5.109843100364008
MA%E Detached: 6.4078969883499
MdA%E Condo Apt: 3.3903548421064453
MA%E Condo Apt: 4.024094983310528
MdA%E Freehold Townhouse: 5.408440560306349
MA%E Freehold Townhouse: 6.646301751245533
MdA%E Condo Townhouse: 4.141791976027536
MA%E Condo Townhouse: 5.214979730938244
MdA%E Semi-Detached: 5.80638119315209
MA%E Semi-Detached: 6.837707844595176
MdA%E Co-Op Apt: 4.813251112741078
MA%E Co-Op Apt: 5.656308038084945
---
Baseline MdA%E list-aware: 7.3
Baseline MA%E list_aware: 8.1


In [1004]:
results = []

month_periods = pd.period_range("2025-12", "2026-05", freq = "M")

for month in month_periods:
    train, valid = homes[homes['sold_month_yr'] < month], homes[homes['sold_month_yr'] == month][valid_cols]

    X_train, y_train = train[off_market_cols + moving_cols].copy(), train['sold_price']
    X_valid, y_valid = valid[off_market_cols].copy(), valid['sold_price']

    cutoff = train['date_sold'].max()

    for col in moving_cols:
        func = lambda row: daily_sales.loc[(daily_sales['date_sold_last'] == min(row['date_listed_before_1d'], cutoff)) &
                                           (daily_sales['house_type_name'] == row['house_type_name']) &
                                           (daily_sales['municipality'] == row['municipality']) &
                                           (daily_sales['house_style'] == row['house_style']), col].values[0]

        X_valid[col] = valid.apply(func, axis = 1)

    X_train['diff_list_cutoff'] = (train['date_listed'] - cutoff).dt.days
    X_valid['diff_list_cutoff'] = (valid['date_listed'] - cutoff).dt.days

    gb_model = models.HistGradientBoostingRegressor(random_state = 0,
                                                    loss = 'squared_error', learning_rate = 0.18,
                                                    max_iter = 400, max_leaf_nodes = 60, min_samples_leaf = 30, max_depth = None,
                                                    l2_regularization = 0.1, max_bins = 255, validation_fraction = None,
                                                    max_features = 1., early_stopping = False, verbose = 1)

    gb_model.fit(X_train, np.log(y_train))

    preds = np.exp(gb_model.predict(X_valid))
    results.append(pd.DataFrame({'predicted': preds, 'y': y_valid, 'house_type': X_valid['house_type_name']}))
    print('---')

print("Off-market training complete.")

results = pd.concat(results).reset_index(drop = True)

Binning 0.049 GB of training data: 0.462 s
Fitting gradient boosted rounds:
Fit 400 trees in 30.759 s, (24000 total leaves)
Time spent computing histograms: 14.548s
Time spent finding best splits:  5.230s
Time spent applying splits:      5.242s
Time spent predicting:           0.365s
---
Binning 0.050 GB of training data: 0.350 s
Fitting gradient boosted rounds:
Fit 400 trees in 27.170 s, (24000 total leaves)
Time spent computing histograms: 13.023s
Time spent finding best splits:  4.425s
Time spent applying splits:      4.414s
Time spent predicting:           0.362s
---
Binning 0.050 GB of training data: 0.412 s
Fitting gradient boosted rounds:
Fit 400 trees in 25.402 s, (24000 total leaves)
Time spent computing histograms: 11.927s
Time spent finding best splits:  4.117s
Time spent applying splits:      4.123s
Time spent predicting:           0.344s
---
Binning 0.051 GB of training data: 0.384 s
Fitting gradient boosted rounds:
Fit 400 trees in 22.659 s, (24000 total leaves)
Time spen

In [1005]:
print("MdA%E: " + str(100 * ((results['predicted'] - results['y']).abs() / results['y']).median()))
print("MA%E: " + str(100 * ((results['predicted'] - results['y']).abs() / results['y']).mean()))
print("Signed MdA%E: " + str(100 * ((results['predicted'] - results['y']) / results['y']).median()))
print("Signed MA%E: " + str(100 * ((results['predicted'] - results['y']) / results['y']).mean()))

print('---')

for ht in results['house_type'].unique():
    results_ht = results[results['house_type'] == ht]

    print(f"MdA%E {ht}: " + str(100 * ((results_ht['predicted'] - results_ht['y']).abs() / results_ht['y']).median()))
    print(f"MA%E {ht}: " + str(100 * ((results_ht['predicted'] - results_ht['y']).abs() / results_ht['y']).mean()))

print('---')

print("Baseline MdA%E off-market: 20.8")
print("Baseline MA%E off_market: 27.4")

MdA%E: 8.187506879980742
MA%E: 10.696690357345307
Signed MdA%E: 0.36978229397848
Signed MA%E: 1.1216566240344998
---
MdA%E Detached: 10.352767889434446
MA%E Detached: 12.972110051837538
MdA%E Condo Apt: 6.864673007486148
MA%E Condo Apt: 9.134108823138327
MdA%E Freehold Townhouse: 9.36080467116249
MA%E Freehold Townhouse: 11.452698764544632
MdA%E Condo Townhouse: 7.331083339848707
MA%E Condo Townhouse: 8.858287197442523
MdA%E Semi-Detached: 9.483768445354684
MA%E Semi-Detached: 11.55716744793042
MdA%E Co-Op Apt: 11.5652481165578
MA%E Co-Op Apt: 18.98807148619043
---
Baseline MdA%E off-market: 20.8
Baseline MA%E off_market: 27.4
